# Thymio project 
This notebook runs the whole system, it connects the Python modules.

| Module | File | Role |
|---|---|---|
| Vision | `vision.py` | map, obstacles, goal and robot pose from the camera |
| Global navigation | `global_nav_lib.py` | shortest path (visibility graph + A*) |
| Filtering | `FilteringnMotionControl/filtering.py` | Kalman filter: fuses odometry and camera |
| Motion control | `FilteringnMotionControl/control.py` | follows the path (wheel commands) |
| Local navigation | `local_navigation/local_navigation.py` | avoids the physical obstacles seen by the proximity sensors |

**Conventions (all modules):** millimetres and radians, origin at the bottom-left
of the map, x to the right, y up, heading 0 = +x, counter-clockwise positive.
The robot pose is the **centre of the body** (ArUco marker), except inside the
Kalman filter, which uses the **wheel axle** (see Pose conversion).

**Order of the cells:** run them from top to bottom, once. The connection cell
must only be run once per kernel.

## Imports
Loads the modules of each part. `%autoreload 2` reloads a `.py` file
automatically when it is saved, so changes in a module are taken into account
without restarting the kernel.

The two module folders are added to `sys.path` so Python can find them.

`thymio.py` is the only file that talks to the robot

In [1]:
%load_ext autoreload
%autoreload 2

import math, sys, time
import numpy as np
import cv2

sys.path += ["FilteringnMotionControl", "local_navigation"]
import config, vision, global_nav_lib
from filtering import ProjectThymio, motion_model, kalman
from conversions import tr_actuators, invtr_actuators, wrap
from local_navigation import LocalNavigation
import local_navigation as ln
from control import MotionController          # TODO (motion control): to create
from config import TS, MAX_UNITS, AXLE_TO_CENTRE_MM, GOAL_RADIUS_MM, KIDNAP_DIST_MM
from thymio import Thymio

ImportError: cannot import name 'MotionController' from 'control' (/Users/learoh/Documents/Mobile_Robotics/BOMR-Project-Group9/FilteringnMotionControl/control.py)

## Parameters 

In [ ]:
MAX_TIME = 180                 # s, safety stop

ALPHAS = np.array([0.01, 25.0, 1e-6, 0.01])     # TODO (filtering): calibrated values
Q0 = np.diag([0.01, 0.01, 1e-6])
R = np.diag([1.0, 1.0, 1e-4])
SIGMA0 = np.diag([1.0, 1.0, 1e-4])

## Connection to the Thymio

In [ ]:
if "robot" not in globals():
    robot = await Thymio.connect()
print("connected to the Thymio")

## Pose conversion

Vision measures the center of the ArUco marker, which is the center of the Thymio. Global and local navigation use that, but filtering uses the wheel axle. 

In [ ]:
def centre_to_axle(pose):
    x, y, th = pose
    return np.array([x - AXLE_TO_CENTRE_MM * math.cos(th),
                     y - AXLE_TO_CENTRE_MM * math.sin(th), th])

def axle_to_centre(mu):
    x, y, th = mu
    return (x + AXLE_TO_CENTRE_MM * math.cos(th),
            y + AXLE_TO_CENTRE_MM * math.sin(th), th)

## Camera, map and first path
The Thymio is standing still. This cell opens the camera and waits until the vision finds the 4 corner markers, the robot and the goal. It keeps the homography, the obstacles and the goal fixed and then plans the first path. `plan(start_xy)` is also used to replan from the current position after a local obstacle or after kidnapping. 

In [ ]:
cap = cv2.VideoCapture(vision.CAMERA_INDEX)

def read_camera(hom):
    """Returns (vision result or None, debug image or None)."""
    ok, frame = cap.read()
    if not ok:
        return None, None
    return vision.process(frame, hom)

def plan(start_xy):
    path, padded, _ = global_nav_lib.plan_path(obstacles, start_xy, goal, config.PADDING_MM,
                                               config.MAP_SIZE, config.WALL_PADDING_MM)
    return path, padded

hom = None
while True:                                   # wait for map + robot + goal
    res, dbg = read_camera(hom)
    if res is not None:
        hom = res["homography"]
        if res["robot"] is not None and res["goal"] is not None:
            break
obstacles, goal, start_pose = res["obstacles"], res["goal"], res["robot"]
path, padded = plan(start_pose[:2])
print("path:", path)

## Live visualisation

The camera image already shows what vision detected: obstacles in magenta,
goal as a red cross, robot pose measured by the camera as an orange arrow
and the global plan shows the enlarged obstacles in cyan and the path with numbered waypoints.
`draw_state` adds:

- **black dot + line:** pose estimated by the Kalman filter and its heading,
- **text:** state of local navigation (`GLOBAL`, or `LOCAL` with its phase and
  side),
- **red circles:** estimated physical obstacle and the circle the robot
  follows around it (radius R).

In [ ]:
def draw_state(img, mu, sigma, cam, local_nav):
    """Filter pose and local navigation state."""
    # filter estimate (axle -> body centre)
    x, y, th = axle_to_centre(mu)
    p = vision.mm_to_px((x, y))
    tip = vision.mm_to_px((x + 60 * math.cos(th), y + 60 * math.sin(th)))
    cv2.circle(img, p, 6, (0, 0, 0), -1)                 # black dot = filter
    cv2.line(img, p, tip, (0, 0, 0), 2)                  # heading

    # local navigation
    colour = (0, 0, 255) if local_nav.state == "LOCAL" else (0, 0, 0)
    text = local_nav.state if local_nav.state == "GLOBAL" else \
        f"LOCAL {local_nav.phase} {local_nav.side}"
    cv2.putText(img, text, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.6, colour, 2)
    if local_nav.centre is not None:
        R = ln.OBSTACLE_RADIUS + ln.ROBOT_HALF_WIDTH + ln.MARGIN
        c = vision.mm_to_px(local_nav.centre)
        cv2.circle(img, c, int(ln.OBSTACLE_RADIUS * vision.PX_PER_MM), (0, 0, 255), 2)
        cv2.circle(img, c, int(R * vision.PX_PER_MM), (0, 0, 255), 1)
    return img

## Main loop

One iteration every `TS` = 0.1 s:

1. The camera image (robot pose or None if hidden) is read. The values of the proximity sensors and the measured wheel speeds are read.
2. Filtering predicts the position with the measured wheel speeds (odometry). If the camera sees the robot, the position is corrected with the camera pose. If the camera pose is very far from the estimate, then it is a kidnapping: the filter is reset to the camera pose and the path is replanned.
3. If the goal is reached, the robot stops.
4. Local navigation has priority. If a front sensor sees an obstacle, it takes control of the wheels and avoids the obstacle. When the obstacle is cleared, the path is replanned.
5. Motion control is only used when local navigation is not active. The robot follows the path.
6. The live view is drawn and the step is saved. 

The loop stops after `MAX_TIME`, or when the middle button of the
Thymio is pressed. Whatever happens (error, interruption), the `finally`
block sets the motors to 0.

In [ ]:
async def run():
    global path, padded
    pt = ProjectThymio(TS, Q0, R, alphas=ALPHAS)
    mu, sigma = centre_to_axle(start_pose), SIGMA0.copy()
    local_nav = LocalNavigation()
    controller = MotionController(path)
    log = []
    t_start = t_prev = time.monotonic()

    try:
        while time.monotonic() - t_start < MAX_TIME:
            t_loop = time.monotonic()

            # 1. sensors
            res, dbg = read_camera(hom)
            cam = res["robot"] if res is not None else None      # body centre, or None
            prox = robot.read_prox()
            u_meas = invtr_actuators(robot.read_wheel_speeds())        # (v, omega) really driven

            # 2. filtering (state = axle)
            pt.Ts, t_prev = t_loop - t_prev, t_loop
            if cam is None:                                       # camera hidden
                mu, sigma = motion_model(mu, sigma, u_meas, pt)
            else:
                z = centre_to_axle(cam)
                z[2] = mu[2] + wrap(z[2] - mu[2])     # same turn as the estimate: no 2π jump
                if math.dist(z[:2], mu[:2]) > KIDNAP_DIST_MM:     # kidnapping
                    mu, sigma = z, SIGMA0.copy()
                    local_nav = LocalNavigation()
                    path, padded = plan(cam[:2])
                    controller = MotionController(path)
                else:
                    mu, sigma = kalman(mu, sigma, u_meas, z, pt)
            mu[2] = wrap(mu[2])
            pose = axle_to_centre(mu)                             # body centre, for navigation

            # 3. goal reached?
            if math.dist(pose[:2], goal) < GOAL_RADIUS_MM:
                print("goal reached")
                break

            # 4. local navigation (has priority)
            was_local = (local_nav.state == "LOCAL")
            state, left, right = local_nav.update(prox, pose, controller.current_target())
            if was_local and state == "GLOBAL":                   # obstacle passed -> replan
                path, padded = plan(pose[:2])
                controller = MotionController(path)

            # 5. global motion control
            if state == "GLOBAL":
                v, omega = controller.step(mu, sigma, t_loop)     # mm/s, rad/s
                left, right = tr_actuators((v, omega))
            left, right = np.clip([left, right], -MAX_UNITS, MAX_UNITS)
            robot.set_wheel_speeds(left, right)

            # 6. visualisation + log
            if dbg is not None:
                vision.draw_plan(dbg, padded, path)
                draw_state(dbg, mu, sigma, cam, local_nav)
                cv2.imshow("Thymio", dbg)
                cv2.waitKey(1)
            log.append(dict(t=t_loop - t_start, mu=mu.copy(), sigma=sigma.copy(),
                            cam=cam, state=state, phase=local_nav.phase,
                            centre=local_nav.centre, left=left, right=right))

            if robot.stop_requested():                        # emergency stop
                break
            await robot.sleep(max(0.0, TS - (time.monotonic() - t_loop)))
    finally:
        robot.stop()
    return log

In [ ]:
# Run
log = await run()

## Plots 

In [ ]:
# trajectory (camera vs filter), LOCAL segments, covariance ellipses, prox over time

## Stop everything

In [ ]:
await robot.close()
del robot
cap.release()
cv2.destroyAllWindows()